<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/PromptEngExercise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip -q install google-generativeai

In [19]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [20]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

model = genai.GenerativeModel("gemini-2.5-flash")

In [21]:
react_prompt = """
You are a Python coding assistant using a ReACT loop. Follow this strict cycle:

Stage 1 — Reason/Plan: Briefly state your plan (max 4 bullet points).
Stage 2 — Act (Generate Code): Output ONLY Python code inside one fenced code block.
Stage 3 — Expected Output: List what outputs/plots should appear (max 3 bullets).

Constraints:
- Use only Python standard libraries + pandas + matplotlib.
- Must run in Google Colab.
- Include basic error handling.
- Do NOT use seaborn.
- Make the code self-contained (generate or embed any data needed).

Task:
Create a small dataset (10–20 rows) representing customer support tickets with columns like
category, priority, resolution_time_minutes. Clean the data (handle missing values), compute
summary stats by category, and make one matplotlib plot.
"""

resp = model.generate_content(react_prompt)
text = resp.text
print(text)

Stage 1 — Reason/Plan:
*   Generate a pandas DataFrame with specified columns and introduce some NaNs.
*   Clean the data by filling missing `resolution_time_minutes` with the median and `category`/`priority` with the mode, then convert types.
*   Calculate the mean `resolution_time_minutes` grouped by `category`.
*   Create a matplotlib bar chart visualizing the average resolution time per category.

Stage 2 — Act (Generate Code):

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def analyze_support_tickets():
    """
    Generates customer support ticket data, cleans it, computes summary stats,
    and creates a matplotlib plot.
    """
    print("--- Starting Ticket Analysis ---")

    # 1. Generate Data
    print("Generating synthetic data...")
    data = {
        'category': ['Technical', 'Billing', 'Technical', 'General', 'Billing',
                     'Technical', 'General', 'Technical', 'Billing', 'General',
                     'Technical', '

In [22]:
import re

def extract_code_block(s: str) -> str:
    match = re.search(r"```python\s*(.*?)```", s, re.DOTALL | re.IGNORECASE)
    if not match:
        match = re.search(r"```\s*(.*?)```", s, re.DOTALL)
    return match.group(1).strip() if match else ""

code = extract_code_block(text)
print("Extracted code length:", len(code))
print(code[:500])

Extracted code length: 3671
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def analyze_support_tickets():
    """
    Generates customer support ticket data, cleans it, computes summary stats,
    and creates a matplotlib plot.
    """
    print("--- Starting Ticket Analysis ---")

    # 1. Generate Data
    print("Generating synthetic data...")
    data = {
        'category': ['Technical', 'Billing', 'Technical', 'General', 'Billing',
                     'Technical', 'General', 'Technical', 'Bil


In [23]:
def run_generated(code_str: str):
    try:
        exec_globals = {}
        exec(code_str, exec_globals)
        return True, ""
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return False, tb

ok, error_trace = run_generated(code)

print("Run success:", ok)
if not ok:
    print("Error trace:\n", error_trace)

Run success: True


In [26]:
# Iteration / Improvement Cycle (ReACT refinement)

improve_prompt = f"""
You previously generated Python code that ran successfully.

Improve the code while keeping it fully runnable in Google Colab.

Requirements:
- Keep using only pandas and matplotlib.
- Keep it self-contained.
- Add one additional basic validation check.
- Improve comments for clarity.
- Output ONLY the improved Python code in one fenced code block.

Here is the original working code:
```python
{code}
"""